In [1]:
!pip install unsloth
!pip install datasets transformers accelerate bitsandbytes trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224

In [2]:
from google.colab import files
files.upload()

Saving dataset.jsonl to dataset.jsonl


{'dataset.jsonl': b'{"instruction": "Who is Yusuf Solomon Olumide?", "output": "Yusuf Solomon Olumide is a Mechatronics Engineering graduate and Machine Learning Engineer who is passionate about robotics, AI, and automation. He graduated from Bells University of Technology with a First Class degree and currently works as a Machine Learning Engineer at Babban Gona."}\r\n{"instruction": "What is Yusuf Solomon\'s educational background?", "output": "Yusuf Solomon holds a Bachelor of Engineering in Mechatronics Engineering from Bells University of Technology, where he graduated with a CGPA of 4.88/5.00. He was ranked in the top 5% of his program, finishing 3rd out of 87 students."}\r\n{"instruction": "Summarize Yusuf Solomon\'s professional experience.", "output": "Yusuf Solomon has experience as a Machine Learning Engineer at Babban Gona, where he trains and monitors models in production. He previously completed a maintenance engineering internship at Seven Up Bottling Company and an IT s

In [5]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
)

==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.3 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [6]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files = "/content/dataset.jsonl",
    split = "train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    }

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/438 [00:00<?, ? examples/s]

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,

    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,

        max_steps = 200,
        learning_rate = 2e-4,

        fp16 = True,
        logging_steps = 10,

        output_dir = "outputs",
        optim = "paged_adamw_8bit",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/438 [00:00<?, ? examples/s]

In [9]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 438 | Num Epochs = 4 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
10,3.301805
20,2.627396
30,2.303768
40,2.168197
50,2.084159
60,1.838444
70,1.602072
80,1.495763
90,1.408561
100,1.484194


TrainOutput(global_step=200, training_loss=1.5328389644622802, metrics={'train_runtime': 218.423, 'train_samples_per_second': 7.325, 'train_steps_per_second': 0.916, 'total_flos': 219903382748160.0, 'train_loss': 1.5328389644622802, 'epoch': 3.6392694063926943})

In [10]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer
)

prompt = """### Instruction:
What can you say about yusuf solomon?

### Response:
"""

print(pipe(prompt, max_new_tokens=100)[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^

### Instruction:
What can you say about yusuf solomon?

### Response:
Yusuf Solomon is an electrical and electronics engineering graduate who is passionate about robotics and AI. He graduated from Bells University of Technology with a First Class degree and has experience in embedded systems, robotics, and AI. He is currently a Machine Learning Engineer at Babban Gona,.


In [13]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer
)

prompt = """### Instruction:
What did solomon study in University

### Response:
"""

print(pipe(prompt, max_new_tokens=100)[0]["generated_text"])

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
What did solomon study in University

### Response:
He studied Mechatronics Engineering at Bells University of Technology in Code Session 2025. His degree included a CGPA of 4.88 out of 5.00. He graduated with a Bachelor of Engineering in Mechatronics Engineering with a Specialization in Industrial Automation,. He also completed a one-year Professional Certificate in Robotics,. He graduated with a CGPA of 4.88 out of 5.00. He graduated with a Bachelor of Engineering in


In [14]:
model.save_pretrained("personal_qwen")
tokenizer.save_pretrained("personal_qwen")

('personal_qwen/tokenizer_config.json', 'personal_qwen/tokenizer.json')

In [15]:
!zip -r personal_qwen.zip personal_qwen

  adding: personal_qwen/ (stored 0%)
  adding: personal_qwen/tokenizer_config.json (deflated 43%)
  adding: personal_qwen/README.md (deflated 65%)
  adding: personal_qwen/tokenizer.json (deflated 81%)
  adding: personal_qwen/adapter_config.json (deflated 58%)
  adding: personal_qwen/adapter_model.safetensors (deflated 7%)


In [16]:
files.download("personal_qwen.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>